In [ ]:
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tensorflow.keras import utils as np_utils

import tensorflow as tf
import numpy as np
import tensorflow_datasets as tfds


In [ ]:
#چون فایل رو دستی دانلود کردیم مجبور به تغییر مسیر هستیم در کد
import pathlib

flowers_root = r'C:\Users\Aryan\.keras\datasets\flower_photos'

flowers_root = pathlib.Path(flowers_root)

In [ ]:
image_count = len(list(flowers_root.glob('*/*.jpg')))
print(image_count)


In [ ]:
# Create a dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
  flowers_root,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(180, 180),
  batch_size= 32)

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
  flowers_root,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(180, 180),
  batch_size= 32)

In [ ]:
class_names = train_ds.class_names
print(class_names)

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_model(conv_blocks, base_filters, kernel_size, dropout_rate, lr, optimizer_name):
    model = models.Sequential()
    model.add(layers.Input(shape=(180, 180, 3)))

    filters = base_filters
    for _ in range(conv_blocks):
        model.add(layers.Conv2D(filters, kernel_size, padding="same", activation="relu"))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D())
        filters *= 2

    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(5))

    if optimizer_name == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )

    return model


In [ ]:
from itertools import product
from tqdm import tqdm
import pandas as pd

learning_rates = [1e-3, 1e-4]
batch_sizes = [32]
conv_blocks = [2]
base_filters = [32]
kernel_sizes = [3]
dropouts = [0.3]
optimizers = ["adam", "sgd"]
epochs_list = [2]


In [ ]:
import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
results = []

search_space = list(product(
    learning_rates,
    batch_sizes,
    conv_blocks,
    base_filters,
    kernel_sizes,
    dropouts,
    optimizers,
    epochs_list
))

for lr, bs, cb, bf, ks, dr, opt, ep in tqdm(search_space):

    model = build_model(cb, bf, ks, dr, lr, opt)

    history = model.fit(
        train_ds.unbatch().batch(bs),
        validation_data=val_ds.unbatch().batch(bs),
        epochs=ep,
        verbose=0
    )

    best_val_acc = max(history.history["val_accuracy"])

    results.append({
        "learning_rate": lr,
        "batch_size": bs,
        "conv_blocks": cb,
        "base_filters": bf,
        "kernel_size": ks,
        "dropout": dr,
        "optimizer": opt,
        "epochs": ep,
        "val_accuracy": best_val_acc
    })

df = pd.DataFrame(results)
df.sort_values("val_accuracy", ascending=False).head(10)


In [ ]:
#بخش ۱ - انتخاب هایپرپارامترها
#در این بخش، هدف تعیین هایپرپارامترهای مناسب برای مدل CNN به منظور دسته‌بندی تصاویر بود. هایپرپارامترهای مختلفی از جمله نرخ یادگیری و بهینه‌ساز مورد بررسی قرار گرفتند.
#برای یافتن بهترین ترکیب، یک جستجوی گرید (Grid Search) کوچک با استفاده از مقادیر مختلف نرخ یادگیری و بهینه‌ساز انجام شد، در حالی که بقیه پارامترها ثابت نگه داشته شدند. هر ترکیب برای ۲ دوره (epoch) روی مجموعه داده آموزشی آموزش داده شد و دقت اعتبارسنجی (validation accuracy) ارزیابی گردید.
#مقادیر تست شده:
#نرخ یادگیری: 0.001 ، 0.0001
#بهینه‌ساز: Adam ، SGD
#اندازه بچ (Batch size): 32
#تعداد بلاک‌های کانولوشن (Conv blocks): 2
#تعداد فیلترهای پایه (Base filters): 32
#اندازه کرنل (Kernel size): 3
#نرخ Dropout: 0.3
#تعداد دوره‌های آموزش (Epochs): 2 (فقط برای مقایسه سریع استفاده شد)
#نتایج نشان داد که بهترین ترکیب به شرح زیر است:
#بهینه‌ساز: Adam

#نرخ یادگیری: 0.001

#این# ترکیب پس از ۲ دوره آموزش، به دقت اعتبارسنجی تقریبی 0.544 (54.4%) دست یافت.

## آنجایی که آموزش عمداً کوتاه (۲ دوره) برای مقایسه هایپرپارامترها انجام شد، انتظار می‌رود دقت پس از آموزش نهایی مدل با تعداددوره‌های بیشتر، بهبود یابد.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# data augmentation

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

])

# مدل جدید با ظرفیت بالاتر برای دقت بیشتر
model = models.Sequential([
    layers.Input((180,180,3)),
    data_augmentation,

    # بلوک ۱
    layers.Conv2D(64, 3, padding='same', activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(0.001)),

    layers.BatchNormalization(),
    layers.MaxPooling2D(),

   # بلوک ۲
    layers.Conv2D(128, 3, padding='same', activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # بلوک ۳ (اضافه شد برای جزئیات بیشتر)
    layers.Conv2D(256, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5)
])

model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# --- Early Stopping ---
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

# --- LR Scheduler ---
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[early_stop, lr_scheduler]
)
# رسم همزمان Loss و Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# نمودار خطا
ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_title('Loss')
ax1.legend()

# نمودار دقت
ax2.plot(history.history['accuracy'], label='Train Acc')
ax2.plot(history.history['val_accuracy'], label='Val Acc')
ax2.set_title('Accuracy')
ax2.legend()

plt.show()




In [ ]:
سimport numpy as np
from sklearn.metrics import classification_report

# ۱. دریافت پیش‌بینی‌ها
y_pred_probs = model.predict(val_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

# ۲. استخراج برچسب‌های واقعی
y_true_list = np.concatenate([y.numpy() for x, y in val_ds], axis=0)

# بررسی هوشمندانه: اگر ۲ بعدی است یعنی One-Hot است، پس argmax کن
if y_true_list.ndim > 1:
    y_true = np.argmax(y_true_list, axis=1)
else:
    y_true = y_true_list

# ۳. چاپ گزارش
print(classification_report(y_true, y_pred))

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# مسیر عکس
img_path = "d:\\test.png"

# خواندن عکس (برای دیدن)
img = tf.keras.utils.load_img(img_path)
plt.imshow(img)
plt.axis("off")  # حذف محورهای عددی
plt.show()

In [ ]:
import numpy as np
from tensorflow.keras.applications.efficientnet import preprocess_input

# تغییر اندازه به 224x224
img = tf.keras.utils.load_img(img_path, target_size=(180, 180))

# تبدیل به آرایه برای مدل
img_array = tf.keras.utils.img_to_array(img)

# افزودن بُعد batch (مدل انتظار دارد ورودی [1,224,224,3] باشد)
img_array = np.expand_dims(img_array, axis=0)

# نرمال‌سازی مخصوص EfficientNet
img_array = preprocess_input(img_array)

In [ ]:
pred = model.predict(img_array)

# پیدا کردن اندیس برترین کلاس
pred_class = np.argmax(pred, axis=1)[0]

# همچنین اعتماد مدل
confidence = np.max(pred)

In [ ]:
plt.imshow(tf.keras.utils.load_img(img_path))
plt.axis("off")
plt.title(f"{class_names[pred_class]} ({confidence*100:.1f}%)")
plt.show()

In [ ]:
#####################################################################################################################################

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# مسیر فایل رو دقیقاً اینجا بده
weights_path = r'D:\Projects\DLCA4\Transfer Learning\efficientnetb0_notop.h5'

# بارگذاری مدل
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights=weights_path,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # فعلاً لایه‌ها را فریز کن

# ساخت مدل نهایی
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

# کامپایل
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("مدل با موفقیت از فایل محلی بارگذاری شد!")

# یادآوری: حتماً دیتاست را با سایز (224, 224) بارگذاری کن.


In [ ]:
# تغییر مدل با اضافه کردن لایه تغییر اندازه (Resizing)
model = models.Sequential([
    layers.Input(shape=(180, 180, 3)), # ابعاد اصلی دیتاست تو
    layers.Resizing(224, 224),         # تغییر سایز به 224x224 برای EfficientNet
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

# دوباره کامپایل کن
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# حالا دوباره model.fit را اجرا کن


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20, # با ۲۰ اپوک شروع کن
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
    ]
)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, precision_score, recall_score

# گرفتن لیبل‌های واقعی
y_true = np.concatenate([y for x, y in val_ds], axis=0)

# پیش‌بینی مدل
y_pred_probs = model.predict(val_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

# محاسبه Precision و Recall (macro برای چندکلاسه)
precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')

print("Precision (macro):", precision)
print("Recall (macro):", recall)

# گزارش کامل
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))


In [ ]:
import matplotlib.pyplot as plt

# Accuracy
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()


In [ ]:
model.summary()


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# مسیر عکس
img_path = "d:\\test.png"

# خواندن عکس (برای دیدن)
img = tf.keras.utils.load_img(img_path)
plt.imshow(img)
plt.axis("off")  # حذف محورهای عددی
plt.show()


In [ ]:
import numpy as np
from tensorflow.keras.applications.efficientnet import preprocess_input

# تغییر اندازه به 224x224
img = tf.keras.utils.load_img(img_path, target_size=(180, 180))

# تبدیل به آرایه برای مدل
img_array = tf.keras.utils.img_to_array(img)

# افزودن بُعد batch (مدل انتظار دارد ورودی [1,224,224,3] باشد)
img_array = np.expand_dims(img_array, axis=0)

# نرمال‌سازی مخصوص EfficientNet
img_array = preprocess_input(img_array)


In [ ]:
pred = model.predict(img_array)

# پیدا کردن اندیس برترین کلاس
pred_class = np.argmax(pred, axis=1)[0]

# همچنین اعتماد مدل
confidence = np.max(pred)


In [ ]:
plt.imshow(tf.keras.utils.load_img(img_path))
plt.axis("off")
plt.title(f"{class_names[pred_class]} ({confidence*100:.1f}%)")
plt.show()


In [ ]:
#########################################################################################################################################

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
import keras_tuner as kt


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])


In [ ]:
def build_model(hp):

    activation = hp.Choice("activation", ["relu","elu"])
    kernel_size = hp.Choice("kernel_size", [3,5])
    l2_value = hp.Choice("l2",[1e-5,1e-4])

    model = models.Sequential([
        layers.Input(shape=(180,180,3)),
        data_augmentation,

        # Block 1
        layers.Conv2D(32, kernel_size, padding='same',
                      kernel_regularizer=regularizers.l2(l2_value),
                      use_bias=False),
        layers.BatchNormalization(),
        layers.Activation(activation),
        layers.MaxPooling2D(),

        # Block 2
        layers.Conv2D(64, kernel_size, padding='same',
                      kernel_regularizer=regularizers.l2(l2_value),
                      use_bias=False),
        layers.BatchNormalization(),
        layers.Activation(activation),
        layers.MaxPooling2D(),

        # Block 3
        layers.Conv2D(128, kernel_size, padding='same',
                      kernel_regularizer=regularizers.l2(l2_value),
                      use_bias=False),
        layers.BatchNormalization(),
        layers.Activation(activation),
        layers.MaxPooling2D(),

        # Block 4
        layers.Conv2D(256, kernel_size, padding='same',
                      kernel_regularizer=regularizers.l2(l2_value),
                      use_bias=False),
        layers.BatchNormalization(),
        layers.Activation(activation),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D()
    ])

    dense_units = hp.Choice("dense_units",[128,256])

    model.add(layers.Dense(dense_units, activation=activation))

    dropout = hp.Float("dropout",0.3,0.5,step=0.1)
    model.add(layers.Dropout(dropout))

    model.add(layers.Dense(5))

    optimizer_name = hp.Choice("optimizer",["adam","rmsprop"])
    lr = hp.Float("learning_rate",1e-4,3e-4,sampling="log")

    if optimizer_name == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    else:
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )

    return model


In [ ]:
tuner = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    directory="bayesian_results",
    project_name="custom_cnn_tuning"
)


In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True
    )
]


In [ ]:
tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)


In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]

for param in best_hp.values:
    print(param, ":", best_hp.get(param))


In [ ]:
best_model = tuner.get_best_models(1)[0]

history = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)


In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label='Train Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend()
plt.title('Loss')

plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# ۱. دریافت پیش‌بینی‌ها
y_pred_probs = model.predict(val_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

# ۲. استخراج برچسب‌های واقعی
y_true_list = np.concatenate([y.numpy() for x, y in val_ds], axis=0)

# بررسی هوشمندانه: اگر ۲ بعدی است یعنی One-Hot است، پس argmax کن
if y_true_list.ndim > 1:
    y_true = np.argmax(y_true_list, axis=1)
else:
    y_true = y_true_list

# ۳. چاپ گزارش
print(classification_report(y_true, y_pred))



In [ ]:
########################################################################################################################################

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import keras_tuner as kt

# ===============================
# Mixup
# ===============================

import tensorflow as tf

def mixup(ds, alpha=0.2):
    def _mix(images, labels):
        # تبدیل لیبل‌ها به float32
        labels = tf.cast(labels, tf.float32)

        batch_size = tf.shape(images)[0]
        lam = tf.random.uniform([], 0, 1)
        index = tf.random.shuffle(tf.range(batch_size))

        mixed_images = lam * images + (1 - lam) * tf.gather(images, index)
        mixed_labels = lam * labels + (1 - lam) * tf.gather(labels, index)
        return mixed_images, mixed_labels

    return ds.map(_mix, num_parallel_calls=tf.data.AUTOTUNE)


# ===============================
# SE Block
# ===============================

def se_block(x, reduction=16):

    filters = x.shape[-1]

    se = layers.GlobalAveragePooling2D()(x)

    se = layers.Dense(filters // reduction, activation="relu")(se)

    se = layers.Dense(filters, activation="sigmoid")(se)

    se = layers.Reshape((1,1,filters))(se)

    x = layers.Multiply()([x,se])

    return x


# ===============================
# Stochastic Depth
# ===============================

class StochasticDepth(layers.Layer):

    def __init__(self, drop_prob):
        super().__init__()
        self.drop_prob = drop_prob

    def call(self, x, training=None):

        if (not training) or self.drop_prob == 0.0:
            return x

        keep_prob = 1 - self.drop_prob

        shape = (tf.shape(x)[0],) + (1,) * (len(x.shape)-1)

        random_tensor = keep_prob + tf.random.uniform(shape)

        binary_tensor = tf.floor(random_tensor)

        return (x / keep_prob) * binary_tensor


# ===============================
# Residual Block
# ===============================

def residual_block(x, filters, kernel_size, activation, l2, drop_prob):

    shortcut = x

    x = layers.Conv2D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False,
        kernel_regularizer=tf.keras.regularizers.l2(l2)
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Activation(activation)(x)

    x = layers.Conv2D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False,
        kernel_regularizer=tf.keras.regularizers.l2(l2)
    )(x)

    x = layers.BatchNormalization()(x)

    x = se_block(x)

    if shortcut.shape[-1] != filters:

        shortcut = layers.Conv2D(filters,1,padding="same",use_bias=False)(shortcut)

    x = StochasticDepth(drop_prob)(x)

    x = layers.Add()([x,shortcut])

    x = layers.Activation(activation)(x)

    return x


# ===============================
# Model Builder for Bayesian
# ===============================

def build_model(hp):

    activation = hp.Choice("activation",["relu","elu"])

    kernel_size = hp.Choice("kernel_size",[3,5])

    l2_value = hp.Choice("l2",[1e-5,1e-4])

    num_blocks = hp.Int("num_blocks",2,4)

    drop_prob = hp.Float("stochastic_depth",0.0,0.2,step=0.1)

    initial_lr = hp.Float("initial_lr",1e-4,5e-4,sampling="log")

    dense_units = hp.Choice("dense_units",[128,256])

    dropout = hp.Float("dropout",0.3,0.5,step=0.1)

    optimizer_name = hp.Choice("optimizer",["adam","rmsprop"])


    inputs = layers.Input(shape=(180,180,3))

    x = layers.RandomFlip("horizontal")(inputs)

    x = layers.RandomRotation(0.1)(x)

    x = layers.RandomZoom(0.1)(x)


    filters = 32

    for i in range(num_blocks):

        x = residual_block(
            x,
            filters,
            kernel_size,
            activation,
            l2_value,
            drop_prob
        )

        x = layers.MaxPooling2D()(x)

        filters *= 2


    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(dense_units,activation=activation)(x)

    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(5,activation="softmax")(x)

    model = tf.keras.Model(inputs,outputs)


    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=initial_lr,
        decay_steps=2000
    )


    if optimizer_name == "adam":
        optimizer = tf.keras.optimizers.Adam(lr_schedule)
    else:
        optimizer = tf.keras.optimizers.RMSprop(lr_schedule)


    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# ===============================
# Bayesian Optimization
# ===============================

tuner = kt.BayesianOptimization(

    build_model,

    objective="val_accuracy",

    max_trials=15,

    directory="bayesian_cnn",

    project_name="resnet_se_model"

)


# ===============================
# Dataset preparation
# ===============================


train_ds = train_ds.map(lambda x, y: (x, tf.one_hot(tf.cast(y, tf.int32), 5)))

train_ds_mix = mixup(train_ds)

val_ds_onehot = val_ds.map(
    lambda x, y: (x, tf.one_hot(tf.cast(y, tf.int32), 5))
)


# ===============================
# Search
# ===============================

tuner.search(
    train_ds_mix,
    validation_data=val_ds_onehot,
    epochs=20
)


# ===============================
# Best model
# ===============================

best_model = tuner.get_best_models(1)[0]

best_model.summary()


# ===============================
# Final evaluation
# ===============================

test_ds_onehot = test_ds.map(lambda x,y: (x, tf.one_hot(y,5)))

best_model.evaluate(test_ds_onehot)

In [ ]:
best_model = tuner.get_best_models(1)[0]
history = best_model.history


In [ ]:
history = best_model.fit(
    train_ds_mix,
    validation_data=val_ds_onehot,
    epochs=20
)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# پیش‌بینی
y_pred_probs = best_model.predict(val_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

# لیبل‌های واقعی
y_true = np.concatenate([y for x, y in val_ds], axis=0)


In [ ]:
print(classification_report(y_true, y_pred))


In [ ]:
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()
